# AIO2026 · M01 · Day 04 — Project 1.2: Chatbot RAG hỏi đáp PDF

**Pipeline RAG (Retrieval-Augmented Generation):**

```
PDF -> chunk -> embedding -> vector DB (ChromaDB) -> retrieve top-k -> prompt(+context) -> LLM -> answer
```

Python thuần, 3 thư viện: `pypdf` (đọc PDF) · `chromadb` (vector DB) · `ollama` (embedding `bge-m3` + LLM `vicuna`).

> Notebook chạy được trên **Google Colab**. Phần cuối đóng gói thành web app Streamlit + mở qua **cloudflared**.

## 0. Cài đặt Ollama trên Colab

In [ ]:
# Cài thư viện hệ thống cần cho Ollama
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils lshw

# Cài Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

# Tắt Flash Attention -> tránh lỗi tương thích GPU trên Colab
os.environ["OLLAMA_FLASH_ATTENTION"] = "false"

# Khởi chạy Ollama server chạy nền, đợi 30s cho server sẵn sàng
subprocess.Popen(["ollama", "serve"])
time.sleep(30)
print("Ollama server đã chạy.")

In [ ]:
# Tải 2 model
!ollama pull bge-m3                 # embedding đa ngôn ngữ, 1024 chiều
!ollama pull vicuna:7b-v1.5-q5_1    # LLM sinh văn bản

## 1. Cài thư viện Python

In [ ]:
!pip install -q pypdf chromadb ollama

import chromadb
import ollama
import pypdf

print("OK")

## 2. Đọc PDF

`extract_text() or ""` xử lý trang chỉ có ảnh (không có text). Thay đường dẫn bằng PDF của bạn
(trên Colab: upload file, hoặc `!gdown <id>` để tải từ Google Drive).

In [ ]:
# !gdown 1abcXYZ...    # (tuỳ chọn) tải PDF mẫu từ Google Drive
PDF_PATH = "./YOLOv10_Tutorials.pdf"  # <-- ĐỔI thành file PDF của bạn

reader = pypdf.PdfReader(PDF_PATH)
full_text = "\n".join(p.extract_text() or "" for p in reader.pages)

print("Số trang :", len(reader.pages))
print("Tổng ký tự:", len(full_text))  # = 0 -> PDF là ảnh scan, cần OCR

## 3. Chunking — cắt text thành đoạn ngắn có overlap

- `chunk_size` quá nhỏ → mất ngữ cảnh; quá lớn → nhiễu. **1000 ký tự** là điểm khởi đầu tốt.
- `overlap` (~10–20%) lặp vài dòng cuối chunk trước ở đầu chunk sau → không cắt đôi câu ở ranh giới.

In [ ]:
def chunk_text(text, size=1000, overlap=200):
    """Cắt text thành đoạn tối đa 'size' ký tự, gối đầu 'overlap' ký tự."""
    paras = [p.strip() for p in text.split("\n") if p.strip()]  # tách đoạn, bỏ dòng trống
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:  # còn chỗ -> gộp tiếp
            cur += p + "\n"
        else:  # đầy -> chốt chunk, mở chunk mới
            if cur:
                chunks.append(cur.strip())
            cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")  # 200 ký tự cuối làm đệm
    if cur.strip():
        chunks.append(cur.strip())
    return chunks


chunks = chunk_text(full_text)
print("Số chunks:", len(chunks))
print("---- chunk[0] (300 ký tự đầu) ----")
print(chunks[0][:300])

## 4. Embedding + lưu Vector Database (ChromaDB)

Embedding = biến text thành vector sao cho 2 đoạn **gần nghĩa** có vector **gần nhau** → tìm theo ý nghĩa.

> Muốn lưu bền ra ổ cứng (khỏi embedding lại): thay `chromadb.Client()` bằng `chromadb.PersistentClient(path="./chroma_db")`.

In [ ]:
def embed(texts):
    """Danh sách text -> danh sách vector (1024 chiều với bge-m3)."""
    return ollama.embed(model="bge-m3", input=texts)["embeddings"]


client = chromadb.Client()  # in-memory: mất khi tắt
collection = client.get_or_create_collection("rag")

collection.add(
    ids=[str(i) for i in range(len(chunks))],  # định danh duy nhất mỗi chunk
    documents=chunks,  # text gốc
    embeddings=embed(chunks),  # vector tương ứng
)
print("Đã index:", collection.count(), "chunks")

## 5. Retrieve — tìm đoạn liên quan nhất

ChromaDB đo **khoảng cách** vector câu hỏi vs mọi vector trong kho, trả `k` chunk gần nhất
(mặc định L2/Euclid; muốn cosine: `metadata={"hnsw:space": "cosine"}` lúc tạo collection).

In [ ]:
def retrieve(query, k=4):
    res = collection.query(query_embeddings=embed([query]), n_results=k)
    return res["documents"][0]


QUERY = "YOLOv10 dùng để làm gì?"
for doc in retrieve(QUERY):
    print(doc[:200])
    print("-" * 40)

## 6. RAG — ghép context + câu hỏi rồi hỏi LLM

2 chi tiết quan trọng: câu **"đừng bịa"** (chống hallucination) và **`temperature=0`** (ổn định, hợp RAG).

In [ ]:
PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh:
{context}

Câu hỏi: {question}

Trả lời:"""


def rag(question, k=4):
    context = "\n\n".join(retrieve(question, k))
    resp = ollama.chat(
        model="vicuna:7b-v1.5-q5_1",
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        options={"temperature": 0},
    )
    return resp["message"]["content"]


print(rag("YOLOv10 là gì?"))

## 7. Thử nghiệm
1. **Có đáp án trong PDF**
2. **Tổng hợp nhiều đoạn**
3. **Không có trong PDF**

In [ ]:
cau_hoi = [
    "YOLOv10 là gì?",  # 1. có trong PDF
    "Liệt kê các cải tiến chính của YOLOv10?",  # 2. tổng hợp nhiều đoạn
    "Thủ đô nước Pháp là gì?",  # 3. KHÔNG có trong PDF -> mong bot nói không biết
]
for q in cau_hoi:
    print("Q:", q)
    print("A:", rag(q))
    print("=" * 60)

## 8. Đóng gói giao diện Streamlit
Ghi code app ra file rồi chạy nền + mở tunnel public.

In [ ]:
%%writefile chatbot_app.py
import streamlit as st
import tempfile, os, time
import pypdf, chromadb, ollama

LLM_MODEL, EMBED_MODEL = "vicuna:7b-v1.5-q5_1", "bge-m3"
PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời:"""

for k, v in {"collection": None, "pdf_name": "", "chat_history": []}.items():
    st.session_state.setdefault(k, v)

def embed(texts):
    return ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"]

def chunk_text(text, size=1000, overlap=200):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:
            cur += p + "\n"
        else:
            if cur:
                chunks.append(cur.strip())
            cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

def process_pdf(uploaded_file):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.getvalue()); path = tmp.name
    text = "\n".join(p.extract_text() or "" for p in pypdf.PdfReader(path).pages)
    os.unlink(path)
    chunks = chunk_text(text)
    col = chromadb.Client().get_or_create_collection(f"rag_{int(time.time())}")
    col.add(ids=[str(i) for i in range(len(chunks))], documents=chunks, embeddings=embed(chunks))
    return col, len(chunks)

def rag(question, collection, k=4):
    res = collection.query(query_embeddings=embed([question]), n_results=k)
    context = "\n\n".join(res["documents"][0])
    resp = ollama.chat(model=LLM_MODEL,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        options={"temperature": 0})
    return resp["message"]["content"]

st.set_page_config(page_title="PDF RAG Chatbot", layout="wide", initial_sidebar_state="expanded")
st.title("PDF RAG Assistant: Native")

with st.sidebar:
    st.subheader("Upload tài liệu")
    f = st.file_uploader("Chọn file PDF", type="pdf")
    if f and st.button("Xử lý PDF", use_container_width=True):
        with st.spinner("Đang xử lý..."):
            st.session_state.collection, n = process_pdf(f)
            st.session_state.pdf_name = f.name
            st.session_state.chat_history = []
        st.success(f"{n} chunks")
    st.info(st.session_state.pdf_name if st.session_state.pdf_name else "Chưa có tài liệu")
    if st.button("Xóa lịch sử chat", use_container_width=True):
        st.session_state.chat_history = []

for m in st.session_state.chat_history:
    with st.chat_message(m["role"]):
        st.write(m["content"])

if st.session_state.collection is None:
    st.info("Upload và xử lý PDF trước khi chat.")
    st.chat_input("Nhập câu hỏi...", disabled=True)
else:
    q = st.chat_input("Nhập câu hỏi của bạn...")
    if q:
        st.session_state.chat_history.append({"role": "user", "content": q})
        with st.chat_message("user"):
            st.write(q)
        with st.chat_message("assistant"):
            with st.spinner("Đang suy nghĩ..."):
                ans = rag(q, st.session_state.collection)
            st.write(ans)
        st.session_state.chat_history.append({"role": "assistant", "content": ans})

In [ ]:
!pip install -q streamlit
# Tải cloudflared (chạy 1 lần)
!wget -q -O cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [ ]:
# Chạy Streamlit nền rồi mở tunnel. Mở link dạng https://....trycloudflare.com để dùng app.
!streamlit run chatbot_app.py --server.port 8501 --server.headless true \
    --server.enableCORS false --server.enableXsrfProtection false &>/content/st.log &
import time

time.sleep(8)
!./cloudflared tunnel --url http://localhost:8501